<a href="https://colab.research.google.com/github/uniqueeee777/ntt-disaster-preparedness/blob/main/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount Google Drive

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#01. Audit Kualitas Data Kebencanaan NTT

Tujuan: Memeriksa struktur, kelengkapan, tipe data, dan konsistensi master dataset sebelum preprocessing.
Unit analisis adalah 22 kabupaten/kota di Provinsi Nusa Tenggara Timur. Master dataset terdiri atas 197 kolom yang mencakup bahaya bencana, kejadian historis, kondisi sosial, infrastruktur, dan program DESTANA.

## A. Pemeriksaan awal

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Memperbaiki lokasi folder sesuai struktur yang terdeteksi di Google Drive Anda
lokasi = Path(
    "/content/drive/MyDrive/NTT_Disaster_Analysis data/"
    "Master_Dataset_Kebencanaan_NTT_Artikel1.xlsx"
)

df = pd.read_excel(lokasi, sheet_name="Master_NTT")

print("Dimensi data:", df.shape)
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dimensi data: (22, 197)


,kabupaten_kota,jenis_wilayah,bps_luas_daerah_2025_km2,bps_jumlah_desa_2024,bps_jumlah_desa_2025,bps_penduduk_2025_ribu,bps_kepadatan_2025_jiwa_per_km2,bps_kepadatan_2026_jiwa_per_km2,bps_persen_miskin_maret_2025,bps_penduduk_miskin_maret_2025_ribu,...,destana_ada_dalam_daftar_raw,inarisk_jiwa_terpapar_agregat_raw,inarisk_kerugian_ekonomi_agregat_raw,inarisk_kerugian_fisik_agregat_raw,inarisk_kerusakan_lingkungan_agregat_raw,inarisk_luas_bahaya_agregat_ha_raw,acebs_rendah_raw,acebs_sedang_raw,acebs_tinggi_raw,acebs_total_raw
0,Sumba Barat,Kabupaten,755.84,74,74,157.60,208,212,26.47,37.08,...,Tidak tercantum,614644,29116.27,1948.216,126998,697099,NaN,NaN,NaN,NaN
1,Sumba Timur,Kabupaten,6975.20,156,200,262.98,38,38,25.64,70.35,...,Ya,1125953,37223.93,4805.809,181067,732791,NaN,NaN,NaN,NaN
2,Kupang,Kabupaten,5130.66,177,177,383.39,75,75,20.32,87.50,...,Ya,624973,17215.07,2330.638,46319,318849,NaN,NaN,NaN,NaN
3,Timor Tengah Selatan,Kabupaten,3931.75,278,278,488.00,124,126,24.50,116.74,...,Ya,1201689,51995.20,6324.885,337918,3139598,NaN,NaN,NaN,NaN
4,Timor Tengah Utara,Kabupaten,2622.29,193,193,279.64,107,108,20.17,53.44,...,Ya,1680827,91185.76,6707.927,496754,2274781,NaN,NaN,NaN,NaN


### Struktur dan duplikasi

In [ ]:

# Struktur data
df.info(verbose=False)

print("\nDistribusi tipe data:")
print(df.dtypes.value_counts())

# Pemeriksaan duplikasi
print("\nDuplikasi baris:", df.duplicated().sum())
print("Duplikasi wilayah:", df["kabupaten_kota"].duplicated().sum())
print("Nama wilayah unik:", df["kabupaten_kota"].nunique())
print("Nama kolom duplikat:", df.columns.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Columns: 197 entries, kabupaten_kota to acebs_total_raw
dtypes: float64(14), int64(131), object(52)
memory usage: 34.0+ KB

Distribusi tipe data:
int64      131
object      52
float64     14
Name: count, dtype: int64

Duplikasi baris: 0
Duplikasi wilayah: 0
Nama wilayah unik: 22
Nama kolom duplikat: 0


## B. Kelengkapan dan tipe data

### Identifikasi nilai kosong dan penanda khusus

In [ ]:

tanda_tidak_tersedia = ["...", "…"]
tanda_garis = ["-", "–", "—"]
semua_penanda = tanda_tidak_tersedia + tanda_garis

kelengkapan = pd.DataFrame({
    "tipe_data": df.dtypes.astype(str),
    "nilai_kosong": df.isna().sum(),
    "tanda_tidak_tersedia": df.isin(tanda_tidak_tersedia).sum(),
    "tanda_garis": df.isin(tanda_garis).sum()
})

kolom_masalah = [
    "nilai_kosong",
    "tanda_tidak_tersedia",
    "tanda_garis"
]

temuan_kelengkapan = kelengkapan.loc[
    kelengkapan[kolom_masalah].sum(axis=1) > 0
]

display(temuan_kelengkapan)


,tipe_data,nilai_kosong,tanda_tidak_tersedia,tanda_garis
bps_garis_kemiskinan_september_2025_rp,object,0,22,0
bps_penduduk_miskin_september_2025_ribu,object,0,22,0
bps_persen_miskin_september_2025,object,0,22,0
bps_jalan_prov_baik_2025_km,object,0,0,1
bps_desa_sinyal_25g_2025,object,0,0,4
bps_desa_sinyal_3g_2025,object,0,0,1
bps_desa_tanpa_sinyal_2025,object,0,0,9
bps_rs_khusus_2025,object,0,0,20
bps_puskesmas_non_rawat_inap_2025,object,0,0,1
bps_desa_ada_disabilitas_eks_kusta_2024,object,0,0,2


### Pemeriksaan keterbacaan numerik
Sebagian kolom bertipe object seharusnya merupakan variabel numerik. Pemeriksaan dilakukan dengan mengabaikan penanda khusus sementara dan menguji konversi nilai yang tersisa.

In [ ]:

# Variabel kategorik
kolom_kategori = [
    "kabupaten_kota",
    "jenis_wilayah",
    "destana_ada_dalam_daftar_raw"
]

kolom_kategori += [
    col for col in df.columns
    if col.startswith("krb_") and col.endswith("_kelas")
]

# Kandidat variabel numerik
kolom_objek = df.select_dtypes(include="object").columns

kandidat_numerik = [
    col for col in kolom_objek
    if col not in kolom_kategori
]

hasil_tipe = []

for col in kandidat_numerik:
    nilai = df[col].dropna()
    nilai = nilai.loc[~nilai.isin(semua_penanda)]
    angka = pd.to_numeric(nilai, errors="coerce")

    hasil_tipe.append({
        "variabel": col,
        "jumlah_numerik": angka.notna().sum(),
        "gagal_konversi": angka.isna().sum()
    })

audit_tipe = pd.DataFrame(hasil_tipe)

print("Variabel kategorik:", len(kolom_kategori))
print("Kandidat numerik bertipe object:", len(kandidat_numerik))

display(audit_tipe)


Variabel kategorik: 14
Kandidat numerik bertipe object: 38


,variabel,jumlah_numerik,gagal_konversi
0,bps_garis_kemiskinan_september_2025_rp,0,0
1,bps_penduduk_miskin_september_2025_ribu,0,0
2,bps_persen_miskin_september_2025,0,0
3,bps_jalan_prov_baik_2025_km,21,0
4,bps_desa_sinyal_25g_2025,18,0
5,bps_desa_sinyal_3g_2025,21,0
6,bps_desa_tanpa_sinyal_2025,13,0
7,bps_rs_khusus_2025,2,0
8,bps_puskesmas_non_rawat_inap_2025,21,0
9,bps_desa_ada_disabilitas_eks_kusta_2024,20,0


## Pemeriksaan konsistensi indikator

### Konsistensi penjumlahan luas bahaya KRB

In [ ]:

jenis_bahaya = [
    col.removeprefix("krb_").removesuffix("_total_ha")
    for col in df.columns
    if col.startswith("krb_") and col.endswith("_total_ha")
]

hasil_krb = []

for jenis in jenis_bahaya:
    awalan = f"krb_{jenis}_"

    komponen = [
        awalan + "rendah_ha",
        awalan + "sedang_ha",
        awalan + "tinggi_ha"
    ]

    luas_kelas = (
        df[komponen]
        .astype("string")
        .replace({"-": "0", "–": "0", "—": "0"})
        .apply(pd.to_numeric, errors="coerce")
    )

    luas_total = pd.to_numeric(
        df[awalan + "total_ha"],
        errors="coerce"
    )

    valid = luas_total.notna() & luas_kelas.notna().all(axis=1)
    selisih = luas_kelas.sum(axis=1) - luas_total

    hasil_krb.append({
        "jenis_bahaya": jenis,
        "wilayah_diperiksa": valid.sum(),
        "tidak_sesuai": (valid & (selisih.abs() > 1)).sum(),
        "selisih_maksimum_ha": selisih[valid].abs().max()
    })

audit_krb = pd.DataFrame(hasil_krb)

display(audit_krb)


,jenis_bahaya,wilayah_diperiksa,tidak_sesuai,selisih_maksimum_ha
0,banjir,21,0,0.0
1,banjir_bandang,21,0,0.0
2,cuaca_ekstrem,21,0,0.0
3,gelombang_abrasi,22,0,0.0
4,gempa_bumi,22,0,0.0
5,karhutla,22,0,0.0
6,kekeringan,22,0,0.0
7,letusan_gunung_api,10,0,0.0
8,tanah_longsor,22,0,0.0
9,tsunami,22,0,0.0


In [ ]:

# Konversi km² menjadi hektare
luas_wilayah = df["bps_luas_daerah_2025_km2"] * 100

kolom_total = [
    col for col in df.columns
    if col.startswith("krb_") and col.endswith("_total_ha")
]

luas_bahaya = df[kolom_total].apply(
    pd.to_numeric, errors="coerce"
)

rasio = luas_bahaya.div(luas_wilayah, axis=0)
rasio["kabupaten_kota"] = df["kabupaten_kota"]

audit_luas = rasio.melt(
    id_vars="kabupaten_kota",
    var_name="jenis_bahaya",
    value_name="rasio_luas"
)

temuan_luas = audit_luas.loc[
    audit_luas["rasio_luas"] > 1
].sort_values("rasio_luas", ascending=False)

print("Jumlah kombinasi wilayah-bahaya yang perlu diperiksa:")
print(len(temuan_luas))

display(temuan_luas)


Jumlah kombinasi wilayah-bahaya yang perlu diperiksa:
59


,kabupaten_kota,jenis_bahaya,rasio_luas
21,Kota Kupang,krb_banjir_total_ha,3.330228
4,Timor Tengah Utara,krb_banjir_total_ha,3.275046
108,Malaka,krb_gempa_bumi_total_ha,1.061217
152,Malaka,krb_kekeringan_total_ha,1.059684
130,Malaka,krb_karhutla_total_ha,1.059459
105,Nagekeo,krb_gempa_bumi_total_ha,1.050273
127,Nagekeo,krb_karhutla_total_ha,1.050123
149,Nagekeo,krb_kekeringan_total_ha,1.049686
103,Sumba Tengah,krb_gempa_bumi_total_ha,1.044687
147,Sumba Tengah,krb_kekeringan_total_ha,1.044670


### Konsistensi indikator desa dan sinyal internet

In [ ]:

kolom_sinyal = [
    "bps_desa_sinyal_4g5g_2025",
    "bps_desa_sinyal_3g_2025",
    "bps_desa_sinyal_25g_2025",
    "bps_desa_tanpa_sinyal_2025"
]

sinyal = (
    df[kolom_sinyal]
    .astype("string")
    .replace({"-": "0", "–": "0", "—": "0"})
    .apply(pd.to_numeric, errors="coerce")
)

jumlah_sinyal = sinyal.sum(
    axis=1, min_count=len(kolom_sinyal)
)

audit_sinyal = pd.DataFrame({
    "kabupaten_kota": df["kabupaten_kota"],
    "jumlah_desa_2025": df["bps_jumlah_desa_2025"],
    "jumlah_menurut_sinyal": jumlah_sinyal
})

audit_sinyal["selisih"] = (
    audit_sinyal["jumlah_menurut_sinyal"]
    - audit_sinyal["jumlah_desa_2025"]
)

display(audit_sinyal)


,kabupaten_kota,jumlah_desa_2025,jumlah_menurut_sinyal,selisih
0,Sumba Barat,74,74,0
1,Sumba Timur,200,200,0
2,Kupang,177,177,0
3,Timor Tengah Selatan,278,278,0
4,Timor Tengah Utara,193,193,0
5,Belu,81,81,0
6,Alor,175,175,0
7,Lembata,151,151,0
8,Flores Timur,250,250,0
9,Sikka,194,194,0


### Konsistensi agregasi kejadian bencana

In [ ]:

hasil_bencana = []

for tahun in range(2021, 2027):
    awalan = f"bencana_{tahun}_"

    kolom_jenis = [
        col for col in df.columns
        if col.startswith(awalan + "jenis_")
        and col.endswith("_baris_raw")
    ]

    jumlah_laporan = df[awalan + "baris_laporan_raw"]
    jumlah_jenis = df[kolom_jenis].sum(axis=1)

    selisih = jumlah_laporan - jumlah_jenis

    kolom_tanggal = (
        awalan + "tanggal_tidak_sesuai_tahun_file_raw"
    )

    hasil_bencana.append({
        "tahun": tahun,
        "jumlah_baris_laporan": jumlah_laporan.sum(),
        "wilayah_selisih_agregasi": (selisih != 0).sum(),
        "tanggal_tidak_sesuai": df[kolom_tanggal].sum()
    })

audit_bencana = pd.DataFrame(hasil_bencana)

display(audit_bencana)


,tahun,jumlah_baris_laporan,wilayah_selisih_agregasi,tanggal_tidak_sesuai
0,2021,128,0,0
1,2022,46,0,1
2,2023,115,0,0
3,2024,54,0,0
4,2025,64,0,0
5,2026,38,0,0


### Pemeriksaan kelengkapan DESTANA

In [ ]:

audit_destana = df[[
    "kabupaten_kota",
    "bps_jumlah_desa_2025",
    "destana_entri_daftar_raw",
    "destana_ada_dalam_daftar_raw"
]].copy()

tidak_tercatat = audit_destana[
    "destana_entri_daftar_raw"
].isna()

melebihi_desa = (
    audit_destana["destana_entri_daftar_raw"]
    > audit_destana["bps_jumlah_desa_2025"]
)

print("Wilayah tidak tercantum:", tidak_tercatat.sum())
print("Wilayah dengan entri melebihi jumlah desa:", melebihi_desa.sum())

display(audit_destana.loc[tidak_tercatat | melebihi_desa])


Wilayah tidak tercantum: 2
Wilayah dengan entri melebihi jumlah desa: 0


,kabupaten_kota,bps_jumlah_desa_2025,destana_entri_daftar_raw,destana_ada_dalam_daftar_raw
0,Sumba Barat,74,NaN,Tidak tercantum
15,Sumba Tengah,65,NaN,Tidak tercantum


## D. Ringkasan audit

In [ ]:

ringkasan = pd.DataFrame({
    "pemeriksaan": [
        "Duplikasi wilayah",
        "Variabel seluruhnya tidak tersedia",
        "Nilai numerik gagal dikonversi",
        "Ketidaksesuaian penjumlahan KRB",
        "Luas bahaya melebihi luas administratif",
        "Ketidaksesuaian jumlah desa menurut sinyal",
        "Ketidaksesuaian agregasi bencana",
        "Tanggal laporan tidak sesuai tahun file",
        "Wilayah tanpa catatan DESTANA"
    ],
    "jumlah_temuan": [
        df["kabupaten_kota"].duplicated().sum(),
        (kelengkapan["tanda_tidak_tersedia"] == len(df)).sum(),
        audit_tipe["gagal_konversi"].sum(),
        audit_krb["tidak_sesuai"].sum(),
        len(temuan_luas),
        (audit_sinyal["selisih"] != 0).sum(),
        audit_bencana["wilayah_selisih_agregasi"].sum(),
        audit_bencana["tanggal_tidak_sesuai"].sum(),
        tidak_tercatat.sum()
    ]
})

display(ringkasan)


,pemeriksaan,jumlah_temuan
0,Duplikasi wilayah,0
1,Variabel seluruhnya tidak tersedia,3
2,Nilai numerik gagal dikonversi,0
3,Ketidaksesuaian penjumlahan KRB,0
4,Luas bahaya melebihi luas administratif,59
5,Ketidaksesuaian jumlah desa menurut sinyal,0
6,Ketidaksesuaian agregasi bencana,0
7,Tanggal laporan tidak sesuai tahun file,1
8,Wilayah tanpa catatan DESTANA,2


# Kesimpulan Audit Kualitas Data

Master dataset mencakup 22 kabupaten/kota dan 197 kolom. Tidak ditemukan duplikasi observasi maupun ketidaksesuaian agregasi aritmetika pada indikator yang diperiksa. Sebanyak 38 variabel numerik yang terbaca sebagai teks dapat dikonversi setelah penanda khusus dikecualikan.

Tiga variabel kemiskinan September 2025 tidak memiliki nilai tersedia. Pemeriksaan luas bahaya KRB menghasilkan 59 kombinasi wilayah dan bahaya dengan luas lebih besar daripada luas administratif BPS. Dua ketidaksesuaian terbesar terdapat pada bahaya banjir Kota Kupang dan Timor Tengah Utara. Selisih ini memerlukan perhatian terhadap kesesuaian definisi spasial dan tidak langsung dikategorikan sebagai kesalahan data.

Agregasi laporan bencana konsisten dengan rincian jenis bencana, tetapi terdapat satu catatan tanggal yang tidak sesuai dengan tahun file 2022. Dua kabupaten tidak tercantum dalam daftar DESTANA, sedangkan data ACeBS hanya tersedia untuk enam wilayah. Temuan tersebut menjadi dasar penanganan data dan seleksi indikator pada tahap preprocessing.